In [69]:
import torch
import wandb
import pandas as pd
from transformers import Trainer, Seq2SeqTrainer, Seq2SeqTrainingArguments

from Levenshtein import distance

import re
import os
from datetime import datetime

os.chdir("../scripts")

from data_processing import poquad, processing
from t5.load_t5 import *

In [70]:
train_df, valid_df = poquad.load_poquad_manually_downloaded("../data/poquad-manually-processed")

In [71]:
train_input = poquad.dataset_into_str_input(train_df)
valid_input = poquad.dataset_into_str_input(valid_df)

In [72]:
valid_input.index += 1

In [73]:
valid_input["is_impossible"] = valid_input["target_text"].str.contains(r"\[BRAK\_ODPOWIEDZI\]")

In [74]:
models_to_evaluate = [
    # ("../models/plt5-small-8epochs", "plt5-small-8epochs"),
    ("plt5-original-small", "plt5-original-small"),
    ("plt5-original-base", "plt5-original-base"),
    ("../models/plt5-small-2epochs", "plt5-small-2epochs"),
    ("../scripts/results/checkpoint-22648", "plt5-small-2epochsV2"),
    ("../scripts/results/checkpoint-45296", "plt5-small-4epochs"),
    ("../scripts/results/checkpoint-67944", "plt5-small-6epochs"),
    # ("../scripts/results/checkpoint-90692", "plt5-small-8epochs")
 ]

In [75]:
gen_texts = pd.read_json("../outputs/plt5-small-8epochs_eval_0.json", orient="index")

In [76]:
eval_df = pd.merge(valid_input, gen_texts, left_index=True, right_index=True)
eval_df = eval_df.rename(columns={0: "gen_text"})

In [77]:
possible_answers_df = eval_df[~eval_df["is_impossible"]].copy()

In [79]:
def rm_answer_prefix(text):
    return re.sub(r"odpowiedź: ", "", text)

In [80]:
possible_answers_df["target_text"] = possible_answers_df["target_text"].apply(rm_answer_prefix)
possible_answers_df["gen_text"] = possible_answers_df["gen_text"].apply(rm_answer_prefix)

In [81]:
result = possible_answers_df[['target_text', 'gen_text']].apply(lambda x: distance(*x)/(len(x.iloc[0]) + len(x.iloc[1])), axis=1).mean()

print("Normalized Levenshtein distance for PLT5-Small trained with 8 Epochs:", result)

Normalized Levenshtein distance for PLT5-Small trained with 8 Epochs: 0.3278126758782814


In [82]:
eval_df[eval_df["is_impossible"]]

,input_text,target_text,is_impossible,gen_text
5,kontekst: Miszna Pisma rabiniczne – w tym Mis...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: Mojżesz
8,kontekst: Emilia Plater Sformowany przez nią ...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: 4 kwietnia
10,kontekst: Emilia Plater Sformowany przez nią ...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: [BRAK_ODPOWIEDZI]
13,kontekst: Peter Phillips Zaręczyny pary ogłos...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: Hello!
20,kontekst: Karnawał Dawniej w zapusty jedzono ...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: placki
...,...,...,...,...
7043,kontekst: Michał Czernecki Od 2013 za dyrekcj...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: [BRAK_ODPOWIEDZI]
7045,kontekst: Michał Czernecki Od 2013 za dyrekcj...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: [BRAK_ODPOWIEDZI]
7050,kontekst: Gimnazjum im. Adama Mickiewicza w Pr...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: działalność kontrrewolucyjną
7051,kontekst: Świnoujście Polską część wyspy Uzna...,odpowiedź: [BRAK_ODPOWIEDZI],True,odpowiedź: [BRAK_ODPOWIEDZI]


In [85]:
# "odpowiedź: [BRAK_ODPOWIEDZI] means that the question is impossible to answer, using is_impossible column and gen_text calculate Bin F1"
def bin_f1_score(df):
    tp = (df["is_impossible"] & df["gen_text"].str.contains(r"\[BRAK\_ODPOWIEDZI\]")).sum()
    fp = ((~df["is_impossible"]) & df["gen_text"].str.contains(r"\[BRAK\_ODPOWIEDZI\]")).sum()
    fn = (df["is_impossible"] & ~df["gen_text"].str.contains(r"\[BRAK\_ODPOWIEDZI\]")).sum()

    print(tp, fp, fn)

    precision = tp / (tp + fp)
    recall = tp / (tp + fn)

    return 2 * (precision * recall) / (precision + recall)



result = bin_f1_score(eval_df)

print("Bin F1 score for PLT5-Small trained with 8 Epochs:", result) # It's Bad

534 1221 762
Bin F1 score for PLT5-Small trained with 8 Epochs: 0.35004916420845616
